In [3]:
import sys, os
sys.path.append(os.path.join('/home/module'))
from Bq_Connect import BQDataProcessor 

/opt/conda/lib/python3.8/site-packages/google/cloud/bigquery_storage_v1/__init__.py:57: FutureWarning: You are using a non-supported Python version (3.8.16).  Google will not post any further updates to google.cloud.bigquery_storage_v1 supporting this Python version. Please upgrade to the latest Python version, or at least to Python 3.9, and then update google.cloud.bigquery_storage_v1.
  warnings.warn(


In [4]:
sql = BQDataProcessor(env="dev") 

[BQDataProcessor] Connected → env=DEV, project=pgd-dev-data-analytics, SA=default (/home/jupyterhub/SA/jupyter-pgd-dev-data-analytics-296ca984dd2f.json)


In [5]:
DEST_TABLE       = "pgd-dev-data-analytics.datamart_audit.dm_det_tbl_trx_nontunai"      

SELECT_QUERY = """
SELECT DISTINCT
    cast (nt.create_date as timestamp)                      AS create_date,
    nt.branch_code                                          AS kode_outlet,
    k.no_kontrak,
    k.cif,
    c.nama,
    nt.no_rek                                               AS no_rek_bank,
    nt.customer_bank_name,
    k.up,
    th.description,
    vt.sub_branch_nm,
    vt.branch_nm,
    vt.area_nm,
    vt.region_nm,
    nt.product_code
FROM `pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_nontunai` nt
LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit` k
    ON nt.id_reff = k.no_kontrak
    AND DATE(nt.create_date) = DATE(k.tgl_transaksi)
LEFT JOIN `pgd-prod-data-analytics.datalake.pegadaian_vt_lookup_branch` vt
    ON CAST(nt.branch_code AS STRING) = CAST(vt.sub_branch_cd AS STRING)
LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_transaction_handler` th
    ON nt.tx_code = th.kode_transaksi
LEFT JOIN `pgd-prod-data-analytics.datalake.pgd_tbl_customer` c
    ON k.cif = c.cif
WHERE
    nt.no_rek IS NOT NULL
    AND DATE(nt.create_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
    AND DATE(nt.create_date) <= CURRENT_DATE()
    AND UPPER(th.description) LIKE '%CAIR%'
"""

In [6]:
sql.execute_ddl(f"""create or replace table {DEST_TABLE} as 
{SELECT_QUERY}
""")

print(sql.read(f"select * from {DEST_TABLE} limit 10"))
print(sql.read(f"select count(*) from {DEST_TABLE} limit 10"))

[DDL] Executing: create or replace table pgd-dev-data-analytics.datamart_audit.dm_det_tbl_trx_nontunai as 

SELECT DISTINCT
    cast (nt....
[DDL] Success.
[READ] Executing query...
[READ] Success → 10 row(s) returned.
                create_date kode_outlet no_kontrak   cif  nama  \
0 2026-01-15 07:26:59+00:00       14395       None  None  None   
1 2026-02-24 07:38:23+00:00       11827       None  None  None   
2 2026-02-24 11:29:27+00:00       11828       None  None  None   
3 2026-01-15 12:49:21+00:00       14395       None  None  None   
4 2026-01-13 04:52:29+00:00       11763       None  None  None   
5 2026-01-15 12:41:58+00:00       11826       None  None  None   
6 2026-01-13 08:44:31+00:00       11827       None  None  None   
7 2026-02-24 10:43:35+00:00       11763       None  None  None   
8 2026-01-15 07:30:57+00:00       11826       None  None  None   
9 2026-01-15 10:30:07+00:00       11827       None  None  None   

       no_rek_bank        customer_bank_name  up      